# Drift-Aware MLOps - Walkthrough

End-to-end demo of the pipeline: load ELEC2, instantiate a model + drift detector, run prequential evaluation, and inspect results.
Run from the repo root with `jupyter notebook notebooks/01_walkthrough.ipynb`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.preprocess import load_processed
from src.drift import build_detector
from src.models import build_model
from src.pipelines.prequential import prequential_run

## 1. Load ELEC2

In [ ]:
X_w, y_w, X_s, y_s = load_processed()
print('warmup:', X_w.shape, 'stream:', X_s.shape, 'class balance:', y_s.mean())

## 2. Run prequential evaluation with HybridDD

In [ ]:
model = build_model('sgd_logistic')
detector = build_detector('hybrid')
res = prequential_run(model, detector, X_s[:10000], y_s[:10000], warmup=200)
print(f'accuracy={res.accuracy:.4f}  drifts={len(res.drift_events)}  retrains={res.retrains}')

## 3. Plot rolling accuracy and drift events

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(res.accuracy_curve, color='#37c5d9', lw=1.0, label='rolling accuracy')
for ev in res.drift_events:
    ax.axvline(ev.index, color='#e07b40', alpha=0.7, lw=0.8)
ax.set_xlabel('sample index')
ax.set_ylabel('rolling accuracy')
ax.legend(loc='lower right')
fig.tight_layout()

## 4. Compare detectors side-by-side

In [ ]:
rows = []
for d_name in ['adwin','ddm','eddm','kswin','page_hinkley','hybrid']:
    m = build_model('sgd_logistic')
    d = build_detector(d_name)
    r = prequential_run(m, d, X_s[:8000], y_s[:8000], warmup=200)
    rows.append({'detector': d_name, 'accuracy': r.accuracy, 'drifts': len(r.drift_events), 'retrains': r.retrains})
pd.DataFrame(rows)